# California Grid Analysis
## Dashboarding Operational Risk in Electricity Demand and Supply

**Author:** Sileshi Hirpa
**Date:** April-May 2026
**Data source:** EIA Form EIA-930 - Balancing Authority Hourly Operations
**Scope:** California balancing authorities only: BANC, CISO, IID, LDWP, TIDC
**Tools:** Python, pandas, Plotly, Jupyter Notebook, Tableau Public

---

## Project Overview

This notebook implements an end-to-end data preparation and analysis pipeline
for publicly available California electricity grid operations data. It loads
hourly EIA-930 balancing authority demand records, validates the California-only
scope, converts timestamps from UTC to Pacific Time, engineers a Stress Index
measuring each hour's demand relative to the observed peak for that balancing
authority, classifies each hour into a review priority tier, and exports a full
processed dataset along with four dashboard-ready files for use in Tableau and
Power BI.

The Stress Index is a custom relative indicator - not a formal operational risk
score - designed to enable fair comparison across California balancing authorities
that differ greatly in scale. The peak demand denominator is derived from the
January-April 2026 dataset window, not from a historical all-time record.

## Important Limitation

This notebook uses publicly available EIA-930 data published by the U.S. Energy
Information Administration. It does not represent any utility's internal
operational systems, enterprise risk models, internal safety frameworks, or
proprietary data pipelines.

---

## Business Question

How can hourly electricity demand data be prepared, validated, and visualized so
that a reviewer can quickly identify periods that may deserve additional
operational attention?

## Project Goals

This notebook demonstrates:

1. Loading and inspecting publicly available hourly electricity demand data.
2. Cleaning and standardizing key fields for consistent downstream use.
3. Validating that the dataset scope matches the California-focused analysis objective.
4. Building time-series visualizations of demand patterns across balancing authorities.
5. Engineering a Stress Index based on demand relative to observed peak demand.
6. Classifying each hour into a review priority tier using the Stress Index.
7. Summarizing the highest-priority hours in a compact, decision-ready table.
8. Exporting processed and dashboard-ready datasets for use in Tableau and Power BI.

## Why This Analysis Matters

Grid reliability depends on understanding when demand approaches observed peak
levels. A reviewer who can quickly identify high-stress hours - and understand
whether local generation or imported energy was meeting that demand - is better
positioned to investigate potential reliability concerns. This notebook provides
that structure through systematic data preparation, feature engineering, and a
priority-ranked output table.

In [ ]:
# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import plotly.express as px
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")

## Step 1 Explanation: Importing Libraries

Before working with the dataset, I import the Python libraries I need for:

- reading the CSV file
- cleaning and transforming the data
- computing simple summary metrics
- creating interactive charts

This is a standard first step in a Jupyter notebook workflow.


In [ ]:
# ============================================
# STEP 2: DEFINE FILE PATHS AND CONSTANTS
# ============================================

DATA_PATH     = "../data/raw/EIA930_BALANCE_2026_Jan_Jun.csv"
PROCESSED_DIR = "../data/processed"
OUTPUT_DIR    = "../outputs"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Review priority classification thresholds
# Named constants make threshold values easy to locate and adjust
HIGH_REVIEW_THRESHOLD   = 90   # Stress Index >= 90  ->  High Review Priority
MEDIUM_REVIEW_THRESHOLD = 75   # Stress Index >= 75  ->  Medium Review Priority
TOP_N_REVIEW_HOURS      = 10   # Rows shown in the ranked review table

print("File paths configured.")
print("Output directories ready.")
print(f"Review thresholds: High >= {HIGH_REVIEW_THRESHOLD}, Medium >= {MEDIUM_REVIEW_THRESHOLD}")
print(f"Top review hours to display: {TOP_N_REVIEW_HOURS}")

## Step 2: File Paths and Configuration

This cell defines:

- the source data file path
- the output directories for processed data and summary tables
- named constants for review priority thresholds and display settings

Using named constants (`HIGH_REVIEW_THRESHOLD`, `MEDIUM_REVIEW_THRESHOLD`,
`TOP_N_REVIEW_HOURS`) makes the classification logic readable and the threshold
values easy to find and adjust without editing the function body directly. They
are defined here - in the configuration section - so any reader encounters them
before the classification step.

The `os.makedirs(exist_ok=True)` calls ensure output directories exist at
runtime, preventing export errors if they have not been created manually.

In [ ]:
# ============================================
# STEP 3: LOAD THE DATA
# ============================================

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully.")
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Preview the first few rows
df.head()

## Step 3: Initial Dataset Inspection

At this stage, no analysis is performed. The goal is to confirm:

- the file loaded without error
- which columns are present in the source data
- what the raw records look like before any transformations
- which fields will be relevant to the California grid analysis

Inspecting the dataset before transforming it surfaces structural issues early -
column name formats, unexpected data types, or missing required fields - before
they affect downstream analysis.

In [ ]:
# ============================================
# STEP 4: RENAME IMPORTANT COLUMNS
# ============================================

rename_dict = {
    "Balancing Authority": "balancing_authority",
    "Data Date": "data_date",
    "Hour Number": "hour_number",
    "UTC Time at End of Hour": "utc_time_at_end_of_hour",
    "Demand Forecast (MW)": "demand_forecast_mw",
    "Demand (MW)": "demand_mw",
    "Net Generation (MW)": "net_generation_mw",
    "Total Interchange (MW)": "total_interchange_mw"
}

df.rename(columns=rename_dict, inplace=True)

print("Columns renamed.")
print(df.columns.tolist())

## Step 4 Explanation: Renaming Columns

Some raw dataset column names are long or contain spaces. I renamed the most important ones because:

- the code becomes easier to read
- typing is easier
- later analysis becomes more consistent


In [ ]:
# ============================================
# STEP 5: BASIC DATA CLEANING
# ============================================

df = df.dropna(subset=["balancing_authority", "utc_time_at_end_of_hour", "demand_mw"])
df = df.drop_duplicates(subset=["balancing_authority", "utc_time_at_end_of_hour"])

df["data_date"] = pd.to_datetime(df["data_date"], errors="coerce")
df["utc_time_at_end_of_hour"] = pd.to_datetime(df["utc_time_at_end_of_hour"], utc=True, errors="coerce")
df["local_time_pacific"] = df["utc_time_at_end_of_hour"].dt.tz_convert("America/Los_Angeles")

numeric_cols = ["demand_mw", "demand_forecast_mw", "net_generation_mw", "total_interchange_mw"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Basic cleaning complete.")
print("New dataset shape:", df.shape)

## Step 5: Data Cleaning and Timezone Conversion

This step performs all required preparation before visualization or feature
engineering:

- **Null removal:** Records missing any of the three required fields -
  balancing authority, UTC timestamp, or demand MW - are dropped.
- **Deduplication:** Rows sharing an identical balancing authority and UTC
  timestamp are removed. Each hour should have at most one record per authority.
- **Timestamp parsing:** The UTC timestamp is parsed as a timezone-aware
  datetime and converted to Pacific Time (`America/Los_Angeles`) via
  `tz_convert()`. This handles PST/PDT transitions automatically.
- **Numeric coercion:** The four primary measurement columns are converted to
  numeric types using `errors="coerce"`, which converts any non-numeric source
  values to NaN rather than raising an error.

Performing these transformations in a single cell ensures that all downstream
steps operate on a consistently structured dataset.

In [ ]:
# ============================================
# STEP 6: REVIEW THE FULL DATASET
# ============================================

print("Summary of the dataset:")
print(df.info())

print("\nBasic statistics:")
df[["demand_mw", "demand_forecast_mw", "net_generation_mw", "total_interchange_mw"]].describe()

## Step 6 Explanation: Reviewing the Full Dataset

Before building visualizations, I review the structure and the summary statistics of the full dataset.

This helps me answer questions such as:

- Are the columns the correct type?
- Are there unusual minimum or maximum values?
- Does anything look suspicious before I start interpreting charts?

This is part of good analytical practice.


In [ ]:
# ============================================
# STEP 7: CREATE THE FIRST FULL-DATA VISUALIZATION
# ============================================

fig_overall = px.line(
    df,
    x="local_time_pacific",
    y="demand_mw",
    title="Hourly Electricity Demand Over Time (Full Dataset)",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)"
    }
)

fig_overall.show()

## Scope Validation: Investigating the Unexpected First Chart

After creating the first demand chart, the visualization showed an unexpected
pattern: most values were compressed near the bottom while a few extreme values
stretched the Y-axis upward significantly.

This kind of visual anomaly is worth investigating before continuing. The cause
is often not the chart itself, but the data scope - specifically, whether the
dataset contains records that do not belong to the intended analysis population.

The next step checks which balancing authorities are actually present in the full
dataset before any scope filtering has been applied.

**Key principle:** A chart can run without error and still be misleading if the
dataset does not match the intended analytical scope. Investigating unexpected
patterns - rather than accepting them at face value - is a core discipline in
data visualization and analytics work.

In [ ]:
# Check unusually high demand rows
df.sort_values("demand_mw", ascending=False)[
    ["balancing_authority", "local_time_pacific", "demand_mw", "demand_forecast_mw", "net_generation_mw", "total_interchange_mw"]
].head(20)

In [ ]:
# Check unusually low demand rows
df.sort_values("demand_mw", ascending=True)[
    ["balancing_authority", "local_time_pacific", "demand_mw", "demand_forecast_mw", "net_generation_mw", "total_interchange_mw"]
].head(20)

## Scope Validation: Checking Balancing Authorities in the Dataset

To understand the unexpected pattern in the first chart, the next step inspects
which balancing authorities are present in the full dataset.

If the file contains authorities from outside California, the chart is answering
a different question than intended: it is showing demand behavior for the entire
United States, not California. That would explain the compressed scale - most
California values would appear small relative to large eastern-grid authorities.

Confirming the dataset scope before continuing ensures that all subsequent
charts and metrics answer the correct business question.

In [ ]:
# ============================================
# STEP 8: CHECK BALANCING AUTHORITIES IN THE DATASET
# ============================================

sorted(df["balancing_authority"].dropna().unique())

## Filtering to California Balancing Authorities Only

After inspecting the list of balancing authorities, the dataset contains records
from across the United States. For this analysis the scope must be limited to
the five California balancing authorities:

| Code | Name |
| --- | --- |
| BANC | Balancing Authority of Northern California |
| CISO | California Independent System Operator |
| IID | Imperial Irrigation District |
| LDWP | Los Angeles Department of Water and Power |
| TIDC | Turlock Irrigation District |

Filtering to these five codes creates `df_ca`, the California-only working
dataframe used for all subsequent analysis steps.

In [ ]:
# ============================================
# STEP 9: FILTER TO CALIFORNIA BALANCING AUTHORITIES
# ============================================

ca_bas = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

df_ca = df[df["balancing_authority"].isin(ca_bas)].copy()

print("Original rows:", len(df))
print("California-only rows:", len(df_ca))
print("\nCalifornia balancing authorities in filtered data:")
print(sorted(df_ca["balancing_authority"].dropna().unique()))

## Why Data Scope Validation Matters

One important lesson from this project was that a chart can run successfully and still be misleading if the dataset scope is wrong.

When I first created the demand visualization, the chart looked unusual. That led me to investigate the data more carefully, and I found that the dataframe still included balancing authorities from outside California. After identifying that issue, I filtered the dataset to California-only balancing authorities before continuing the analysis.

This reflects an important analytical habit:
before interpreting a chart, I should confirm that the data actually matches the intended business question.


In [ ]:
# ============================================
# STEP 10: RECHECK DEMAND STATISTICS FOR CALIFORNIA-ONLY DATA
# ============================================

df_ca["demand_mw"].describe()

## Step 10 Explanation: Recheck the California-Only Distribution

After filtering the dataset to California balancing authorities, I checked the `demand_mw` summary statistics again.

### Why this step is useful
Earlier, the full dataset contained unusually large and negative values. After filtering to California only, the values should be more reasonable and more useful for charting.

This check helps confirm that the filtering step improved the dataset for the intended analysis.


In [ ]:
# ============================================
# STEP 11: REBUILD THE DEMAND CHART USING CALIFORNIA-ONLY DATA
# ============================================

fig_ca_single = px.line(
    df_ca,
    x="local_time_pacific",
    y="demand_mw",
    title="Hourly Electricity Demand Over Time (California Only)",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)"
    }
)

fig_ca_single.show()

## Step 11 Explanation: Improve the California-Only Visualization

After rebuilding the line chart with the California-only dataframe, the chart is already more reasonable than the original full-data chart.

However, the California subset still contains multiple balancing authorities. If I draw them as one combined series, the chart can still be misleading. The next step is to separate the lines by balancing authority.


In [ ]:
# ============================================
# STEP 12: COMBINED COMPARISON CHART BY BALANCING AUTHORITY
# ============================================

fig_compare = px.line(
    df_ca.sort_values(["balancing_authority", "local_time_pacific"]),
    x="local_time_pacific",
    y="demand_mw",
    color="balancing_authority",
    title="Hourly Electricity Demand Over Time by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_compare.show()

## Step 12 Explanation: Keep the Combined Comparison Chart

This chart is useful because it shows:

- all California balancing authorities in one view
- their relative scale differences
- that CISO is much larger than the others

This makes the chart useful for big-picture comparison.


In [ ]:
# ============================================
# STEP 13: PANEL VIEW FOR BETTER READABILITY
# ============================================

fig_panel = px.line(
    df_ca.sort_values(["balancing_authority", "local_time_pacific"]),
    x="local_time_pacific",
    y="demand_mw",
    facet_row="balancing_authority",
    title="Hourly Electricity Demand Over Time by California Balancing Authority (Panel View)",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1000
)

fig_panel.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_panel.show()

## Step 13 Explanation: Keep the Panel Chart for Readability

The combined chart is useful for comparison, but it is not ideal for closely viewing the smaller balancing authorities because the largest series dominates the scale.

The panel chart solves that problem by giving each balancing authority its own space. This makes smaller series easier to read and easier to explain.


## Step 14: Key Patterns in California Grid Demand

After creating both the combined chart and the panel chart, the main observations
are:

- **CISO dominates by scale.** The California ISO operates at demand levels far
  exceeding the other four balancing authorities. In a combined view, CISO's
  scale compresses all smaller series. A panel view is required to meaningfully
  review smaller authorities.
- **Daily demand cycles are clearly visible.** All authorities show repeating
  time-based patterns consistent with daily variation in electricity consumption.
- **Smaller authorities have distinct profiles.** IID and TIDC show different
  peak-hour timing from CISO and BANC. A combined view would obscure these
  differences; per-authority analysis is essential for fair comparison.

### Why two chart types are more useful than one

| Chart type | Best use |
| --- | --- |
| Combined line chart | Cross-authority scale comparison |
| Faceted panel chart | Per-authority pattern readability |

Keeping both visualizations allows different analytical questions to be answered
without requiring the reviewer to switch between separate views.

In [ ]:
# ============================================
# STEP 15: FEATURE ENGINEERING - STRESS INDEX
# ============================================

# Compute per-authority peak demand from the dataset window.
# Note: peak_demand_mw is derived from the Jan-Apr 2026 dataset window only,
# not a historical all-time peak. Values reflect relative demand behavior
# within this period and should not be compared against multi-year records.
peak_demand_ca = (
    df_ca.groupby("balancing_authority")["demand_mw"]
    .max()
    .reset_index(name="peak_demand_mw")
)

# Drop existing column if present (ensures idempotent cell reruns)
df_ca = df_ca.drop(columns=["peak_demand_mw"], errors="ignore")

# Merge peak demand back into the California dataframe
df_ca = df_ca.merge(peak_demand_ca, on="balancing_authority", how="left")

# Stress Index: each hour's demand as a percentage of that authority's observed peak
df_ca["stress_index"] = (df_ca["demand_mw"] / df_ca["peak_demand_mw"]) * 100
df_ca["stress_index"] = df_ca["stress_index"].round(2)

df_ca[["balancing_authority", "local_time_pacific", "demand_mw", "peak_demand_mw", "stress_index"]].head()

## Feature Engineering: Stress Index Calculation

The Stress Index is the central analytical metric of this project.

**Formula:** `stress_index = (demand_mw / peak_demand_mw) × 100`

**Why a relative index rather than raw demand values?**

California's five balancing authorities differ greatly in scale. CISO operates
at tens of thousands of megawatts; smaller authorities such as IID and TIDC
operate at hundreds of megawatts. A fixed MW threshold applied uniformly would
systematically miss stress events at smaller authorities. The Stress Index
normalizes each authority against its own observed peak, enabling meaningful
comparison across scales.

**Peak demand denominator:**
`peak_demand_mw` is the maximum `demand_mw` observed for each balancing
authority within the January-April 2026 dataset window. It is computed from
`df_ca` via a `groupby` aggregation and is constant for all rows sharing the
same balancing authority code.

**Dataset window limitation:**
The Stress Index denominator is derived from this dataset window only - not a
historical all-time peak. Index values should be interpreted as relative
indicators within this dataset, not as comparisons against multi-year records.

In [ ]:
# ============================================
# STEP 16: REVIEW PRIORITY CLASSIFICATION
# ============================================

def label_stress(x):
    if x >= HIGH_REVIEW_THRESHOLD:
        return "High Review Priority"
    elif x >= MEDIUM_REVIEW_THRESHOLD:
        return "Medium Review Priority"
    else:
        return "Low Review Priority"

df_ca["review_priority"] = df_ca["stress_index"].apply(label_stress)

df_ca["review_priority"].value_counts()

## Review Priority Classification

The Stress Index is a continuous metric (0-100). Converting it into categorical
tiers makes the results easier to communicate and filter in a dashboard.

**Tier definitions:**

| Label | Stress Index Range | Interpretation |
| --- | --- | --- |
| High Review Priority | >= `HIGH_REVIEW_THRESHOLD` (90) | Demand at or near the highest observed level for that authority |
| Medium Review Priority | >= `MEDIUM_REVIEW_THRESHOLD` (75) and < 90 | Elevated demand worth tracking in trend context |
| Low Review Priority | < 75 | Normal operating range for this dataset window |

The thresholds are defined as named constants at the top of the notebook and
referenced here by name. This makes the classification logic easy to locate and
adjust without editing the function body directly.

## Key Findings: Review Priority Distribution

The following summary is computed directly from `df_ca` after Stress Index
classification. These figures provide a reproducible audit trail for the
statistics cited in the project README.

In [ ]:
# ============================================
# KEY FINDINGS SUMMARY
# ============================================

total_records = len(df_ca)
priority_counts = df_ca["review_priority"].value_counts()

high_count   = priority_counts.get("High Review Priority", 0)
medium_count = priority_counts.get("Medium Review Priority", 0)
low_count    = priority_counts.get("Low Review Priority", 0)

high_pct   = round((high_count   / total_records) * 100, 2)
medium_pct = round((medium_count / total_records) * 100, 2)
low_pct    = round((low_count    / total_records) * 100, 2)

peak_row    = df_ca.loc[df_ca["demand_mw"].idxmax()]
peak_ba     = peak_row["balancing_authority"]
peak_demand = peak_row["demand_mw"]

avg_stress = round(df_ca["stress_index"].mean(), 1)

print("=== Key Findings: California Grid Dataset (Jan-Apr 2026) ===")
print(f"Total hourly records:           {total_records:,}")
print(f"High Review Priority hours:     {high_count:,}  ({high_pct:.2f}% of total)")
print(f"Medium Review Priority hours:   {medium_count:,}  ({medium_pct:.2f}% of total)")
print(f"Low Review Priority hours:      {low_count:,}  ({low_pct:.2f}% of total)")
print(f"Peak demand (single hour):      {peak_demand:,.0f} MW  -  {peak_ba}")
print(f"Average Stress Index (all BAs): {avg_stress}%")

In [ ]:
# ============================================
# STEP 17: IDENTIFY THE TOP HOURS FOR REVIEW
# ============================================

top_review_hours = (
    df_ca.sort_values("stress_index", ascending=False)
         .loc[:, [
             "local_time_pacific",
             "balancing_authority",
             "demand_mw",
             "peak_demand_mw",
             "stress_index",
             "review_priority",
             "net_generation_mw",
             "total_interchange_mw"
         ]]
         .head(TOP_N_REVIEW_HOURS)
         .reset_index(drop=True)
)

top_review_hours.index += 1
top_review_hours.index.name = "Rank"

top_review_hours

## Step 17: Building the Review Summary Table

A decision-maker often needs a short, ranked list rather than a full dataset.

This step produces a compact review table of the `TOP_N_REVIEW_HOURS` hours
with the highest Stress Index values across all California balancing authorities.
Including `net_generation_mw` and `total_interchange_mw` in the table provides
operational context alongside the demand and stress values.

In [ ]:
# ============================================
# STEP 18: VISUALIZE THE TOP REVIEW HOURS
# ============================================

fig_top10 = px.bar(
    top_review_hours.reset_index().sort_values("stress_index", ascending=False),
    x="Rank",
    y="stress_index",
    color="balancing_authority",
    hover_data=[
        "local_time_pacific",
        "demand_mw",
        "peak_demand_mw",
        "review_priority",
        "net_generation_mw",
        "total_interchange_mw"
    ],
    title="Top 10 Hours for Additional Review in California",
    labels={
        "Rank": "Review Rank",
        "stress_index": "Stress Index",
        "balancing_authority": "Balancing Authority"
    },
    text="stress_index",
    width=1200,
    height=500
)

fig_top10.show()

## Step 18: Interpreting the Top Review Hours

The review table and bar chart show the hours in the California-only dataset
that were closest to each balancing authority's observed peak demand during
the January-April 2026 period.

### Key observations
- All top-ranked rows have Stress Index values close to 100, meaning they
  occurred at or very near each authority's observed peak demand.
- Multiple balancing authorities appear in the table - LDWP, BANC, IID, CISO,
  and TIDC - because the Stress Index normalizes each authority against its own
  peak, enabling fair cross-authority comparison.
- If only raw MW values were used to rank hours, CISO would dominate the entire
  list and peak-stress events at smaller authorities would not appear.

### Design choice: relative vs. absolute ranking

A raw demand ranking surfaces only CISO hours in the top positions, providing
no information about stress levels at BANC, IID, LDWP, or TIDC. The Stress
Index solves this by anchoring each hour to that authority's own observed peak,
making the ranking meaningful across authorities of very different operating
scales.

## Step 19: Operational Context - Supporting Multi-Dimension Review

The Stress Index and review priority fields support review from multiple
analytical perspectives. The supporting fields in the dataset -
`net_generation_mw` and `total_interchange_mw` - provide operational context
for each reviewed hour.

### Available analytical dimensions

- **Demand stress view:** Hours with Stress Index values near 100 represent
  periods when demand approached the highest observed level for that authority
  within the dataset window.
- **Generation adequacy view:** Comparing `demand_mw` against
  `net_generation_mw` shows whether local generation was sufficient to meet
  demand, or whether the authority was a net importer during that hour.
- **Interchange view:** `total_interchange_mw` indicates whether the authority
  was a net importer (negative) or exporter (positive) of energy during periods
  of elevated demand.
- **Priority queue view:** The review priority table narrows a 13,020-row
  dataset into a short, ranked list of high-stress hours suitable for further
  investigation.

### Analytical principle

A well-structured data product reduces the time a reviewer spends locating
relevant records. The Stress Index and priority tier translate raw demand values
into a review-ready signal - organizing the data so that the most operationally
notable hours appear at the top of the output, regardless of which balancing
authority they belong to.

In [ ]:
# ============================================
# STEP 20: SAVE FINAL OUTPUT TABLES
# ============================================

# Summary table — review-priority export goes to outputs/
top_review_hours.to_csv(f"{OUTPUT_DIR}/top_review_hours_california.csv", index=True)

# Full cleaned dataset — processed data goes to data/processed/
df_ca.to_csv(f"{PROCESSED_DIR}/cleaned_california_grid_data.csv", index=False)

print("Saved final California-only output tables successfully.")

## Step 20: Exporting Processed Outputs

Two files are exported at this stage:

1. **`data/processed/cleaned_california_grid_data.csv`** - The full
   California-only processed dataset. Retains all columns from the EIA source
   plus the engineered fields (`local_time_pacific`, `stress_index`,
   `review_priority`, `peak_demand_mw`). This is the input for the Phase 4
   dashboard data model build that follows.
2. **`outputs/top_review_hours_california.csv`** - The top review hours table
   showing the highest-stress hourly records ranked by Stress Index.

Both files are excluded from version control by `.gitignore`. The `data/README.md`
file documents the download and reproduction instructions for any reviewer who
clones the repository.

---

## Phase 4: Dashboard Data Model Export

The four files generated in this section are the primary data sources for the
Tableau dashboard and the planned Power BI implementation. They replace the wide
69-column processed file with purpose-built datasets sized and structured for
fast dashboard queries.

| File | Rows | Columns | Role |
| --- | --- | --- | --- |
| `california_grid_dashboard_ready.csv` | 13,020 | 9 | Hourly detail - primary Tableau source |
| `california_grid_monthly_summary.csv` | 20 | 11 | Month-level trend charts |
| `california_grid_hourly_risk_summary.csv` | 120 | 10 | Hour-of-day stress profile |
| `california_grid_kpi_summary.csv` | 5 | 12 | KPI cards - one row per balancing authority |

All four files are excluded from version control by `.gitignore` and must be
regenerated locally by running this notebook from top to bottom.

In [ ]:
# ============================================
# PHASE 4: DASHBOARD DATA MODEL EXPORT
# ============================================

# --- File 1: Dashboard-ready hourly detail (9 columns) ---
DASHBOARD_COLS = [
    "balancing_authority", "local_time_pacific", "demand_mw",
    "demand_forecast_mw", "net_generation_mw", "total_interchange_mw",
    "stress_index", "review_priority", "peak_demand_mw"
]

df_dashboard = df_ca[DASHBOARD_COLS].copy()
dashboard_path = f"{PROCESSED_DIR}/california_grid_dashboard_ready.csv"
df_dashboard.to_csv(dashboard_path, index=False)
print(f"Saved: {dashboard_path}  |  shape: {df_dashboard.shape}")

# --- File 2: Monthly summary (5 BAs x 4 months = 20 rows) ---
df_ca["year_month"] = df_ca["local_time_pacific"].dt.strftime("%Y-%m")

monthly_summary = (
    df_ca.groupby(["balancing_authority", "year_month"])
    .agg(
        avg_demand_mw         = ("demand_mw",       "mean"),
        max_demand_mw         = ("demand_mw",       "max"),
        min_demand_mw         = ("demand_mw",       "min"),
        avg_stress_index      = ("stress_index",    "mean"),
        max_stress_index      = ("stress_index",    "max"),
        total_hours           = ("demand_mw",       "count"),
        high_priority_hours   = ("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_hours = ("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_hours    = ("review_priority", lambda x: (x == "Low Review Priority").sum()),
    )
    .reset_index()
)
for col in ["avg_demand_mw", "avg_stress_index", "max_stress_index"]:
    monthly_summary[col] = monthly_summary[col].round(2)

monthly_path = f"{PROCESSED_DIR}/california_grid_monthly_summary.csv"
monthly_summary.to_csv(monthly_path, index=False)
print(f"Saved: {monthly_path}  |  shape: {monthly_summary.shape}")

# --- File 3: Hourly risk summary (5 BAs x 24 hours = 120 rows) ---
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

hourly_risk = (
    df_ca.groupby(["balancing_authority", "hour_of_day"])
    .agg(
        avg_demand_mw         = ("demand_mw",       "mean"),
        max_demand_mw         = ("demand_mw",       "max"),
        avg_stress_index      = ("stress_index",    "mean"),
        max_stress_index      = ("stress_index",    "max"),
        total_observations    = ("demand_mw",       "count"),
        high_priority_count   = ("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_count = ("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_count    = ("review_priority", lambda x: (x == "Low Review Priority").sum()),
    )
    .reset_index()
)
for col in ["avg_demand_mw", "avg_stress_index", "max_stress_index"]:
    hourly_risk[col] = hourly_risk[col].round(2)

hourly_path = f"{PROCESSED_DIR}/california_grid_hourly_risk_summary.csv"
hourly_risk.to_csv(hourly_path, index=False)
print(f"Saved: {hourly_path}  |  shape: {hourly_risk.shape}")

# --- File 4: KPI summary (1 row per balancing authority) ---
kpi_summary = (
    df_ca.groupby("balancing_authority")
    .agg(
        peak_demand_mw        = ("demand_mw",           "max"),
        avg_demand_mw         = ("demand_mw",           "mean"),
        avg_stress_index      = ("stress_index",        "mean"),
        total_hours           = ("demand_mw",           "count"),
        date_range_start      = ("local_time_pacific",  "min"),
        date_range_end        = ("local_time_pacific",  "max"),
        high_priority_hours   = ("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_hours = ("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_hours    = ("review_priority", lambda x: (x == "Low Review Priority").sum()),
    )
    .reset_index()
)
kpi_summary["avg_demand_mw"]       = kpi_summary["avg_demand_mw"].round(2)
kpi_summary["avg_stress_index"]    = kpi_summary["avg_stress_index"].round(2)
kpi_summary["pct_high_priority"]   = (
    kpi_summary["high_priority_hours"] / kpi_summary["total_hours"] * 100
).round(4)
kpi_summary["pct_medium_priority"] = (
    kpi_summary["medium_priority_hours"] / kpi_summary["total_hours"] * 100
).round(4)
kpi_summary["date_range_start"] = kpi_summary["date_range_start"].astype(str)
kpi_summary["date_range_end"]   = kpi_summary["date_range_end"].astype(str)

kpi_path = f"{PROCESSED_DIR}/california_grid_kpi_summary.csv"
kpi_summary.to_csv(kpi_path, index=False)
print(f"Saved: {kpi_path}  |  shape: {kpi_summary.shape}")

print()
print("All four dashboard-ready files exported successfully.")
print("Note: These files are excluded from version control by .gitignore.")

### Phase 4 Export Notes

- **`california_grid_dashboard_ready.csv`** selects the 9 columns needed by
  Tableau and Power BI, excluding the remaining EIA source columns from the wide
  69-column processed file. This reduces the data source from ~3.6 MB to ~1.1 MB.
- **`california_grid_monthly_summary.csv`** pre-aggregates demand and stress
  statistics by balancing authority and calendar month. Connecting Tableau to
  this file for monthly trend views avoids aggregating the full 13,020-row
  detail file at dashboard query time.
- **`california_grid_hourly_risk_summary.csv`** provides an hour-of-day demand
  and stress profile, suitable for a heatmap or bar chart showing when demand
  stress peaks across the daily cycle.
- **`california_grid_kpi_summary.csv`** provides one summary row per balancing
  authority for use in KPI card visuals in Tableau and Power BI.

The `year_month` and `hour_of_day` fields added during this phase remain in
`df_ca` for reference but are not included in the 9-column dashboard-ready
export.

## Step 21: Analytical Workflow Summary

The notebook implements a structured, reproducible pipeline from raw data
ingestion to dashboard-ready exports:

1. **Load and inspect** the source EIA-930 dataset to understand its structure
   before transforming it.
2. **Clean and standardize** key fields: drop incomplete records, remove
   duplicates, convert UTC timestamps to Pacific Time, and coerce numeric
   columns.
3. **Validate the scope** - the raw file contains balancing authorities from
   across the United States; the notebook identifies this issue via an
   unexpected chart pattern and corrects it by filtering to California-only
   records.
4. **Rebuild visualizations** using the corrected scope, in both combined and
   per-authority panel formats.
5. **Engineer the Stress Index** using an idempotent merge that computes
   per-authority peak demand and expresses each hour's demand as a percentage
   of that peak.
6. **Classify review priorities** using named threshold constants, converting
   the continuous Stress Index into a three-tier categorical label.
7. **Summarize key findings** directly from the data, providing a reproducible
   audit trail for the statistics cited in the project README.
8. **Export the full processed dataset** and a ranked review table.
9. **Build the dashboard data model** - four dashboard-ready CSV files
   optimized for Tableau and Power BI.

Every output file is excluded from version control. Any reviewer with the raw
EIA data can run this notebook top-to-bottom to reproduce all outputs.

## Explore the Interactive Result

This notebook documents the analytical workflow behind the project, including data preparation, California-only validation, time conversion, visualization design, and the creation of the Stress Index used for review prioritization.

To see the final interactive visualization product, please explore the Tableau Public dashboard linked below. The dashboard extends the notebook results into a more visual decision-support format focused on **grid reliability**, **operational review**, and **risk-informed interpretation**.

**Interactive Tableau Dashboard:**  
[Explore the Interactive Result on Tableau Public](https://public.tableau.com/app/profile/sileshi.hirpa1285/viz/CAGridOperationalRiskReliabilityDashboard/riskOverview#2)

---

## Limitations and Caveats

**Dataset window**
The analysis covers January-April 2026. The raw EIA file is named
`EIA930_BALANCE_2026_Jan_Jun.csv` - it was downloaded before full-year data was
available for the 2026 period. All Stress Index values and summary statistics
reflect data from this specific window only.

**Stress Index is not a historical all-time peak**
The `peak_demand_mw` denominator is the highest demand observed for each
balancing authority within the January-April 2026 window. A Stress Index of 100
means the hour matched the highest demand recorded in this window - it may not
correspond to an absolute all-time peak for that authority.

**Public data only**
This project uses EIA Form EIA-930 data published by the U.S. Energy Information
Administration (public domain). It does not represent any utility's internal
operational systems, enterprise risk assessments, consequence models, or
proprietary data pipelines.

**Stress Index is a custom review tool**
The Stress Index is a purpose-built relative indicator for this project. It is
not a formal operational risk score and does not replicate any utility's internal
methodology.

**No real-time data connection**
The dataset is static. This pipeline does not automatically refresh as new EIA
data becomes available.

**No consequence or impact modeling**
This project identifies periods of elevated demand relative to observed peak
levels. It does not model the probability, cause, or consequence of reliability
events.

**Power BI version is planned**
The planned Power BI implementation will use the existing dashboard data model.
It will be added to the portfolio when complete.

## Conclusion and Portfolio Summary

### What This Notebook Demonstrates

This notebook implements an end-to-end data analysis pipeline for California
electricity grid operations data. Starting from a raw 27 MB EIA-930 file, it:

- **Loads and inspects** publicly available hourly balancing authority data.
- **Validates the analytical scope** - explicitly identifying and correcting a
  misleading chart caused by non-California data remaining in the dataset before
  filtering.
- **Cleans and prepares** the data with UTC-to-Pacific timezone conversion, null
  removal, deduplication, and numeric type coercion.
- **Engineers a Stress Index** that normalizes each hour's demand against the
  highest observed demand for that authority, enabling fair cross-authority
  comparison regardless of operating scale.
- **Classifies review priorities** using named threshold constants, converting
  the Stress Index into a three-tier decision-support label.
- **Computes key findings** directly from the data, including priority hour
  counts and per-authority peak demand values.
- **Exports a full processed dataset** and four dashboard-ready files structured
  for Tableau and Power BI use.

### Why Scope Validation Was the Most Important Step

The scope validation step (Steps 7-9) is the most analytically significant part
of this workflow. The first demand chart ran without error but produced a
misleading visualization because the dataset still contained balancing authorities
from across the United States. Recognizing that an error-free chart can still be
analytically incorrect, and correcting that issue before continuing, reflects a
core discipline in data preparation and visualization engineering.

### Stress Index as a Relative Review Indicator

The Stress Index normalizes each authority's demand against its own observed peak
rather than against a shared absolute threshold. This design ensures that smaller
authorities such as IID and TIDC receive fair representation in the review
priority output - a uniform MW threshold would systematically suppress stress
signals at smaller-scale authorities.

The index is a custom tool for this project. It uses a dataset-window peak rather
than a historical all-time record and should be interpreted as a relative
indicator within this dataset only.

### Connection to the Dashboard Layer

The four CSV files exported in Phase 4 are the primary data sources for the
Tableau dashboard. The planned Power BI implementation will use the existing
dashboard data model:

- The 9-column hourly detail file supports time-series views, the review queue,
  and filter interaction.
- The monthly and hourly summary files support pre-aggregated trend and profile
  views without requiring the dashboard tool to aggregate the full detail file
  at query time.
- The KPI summary file provides per-authority statistics for scorecard visuals.

### Future Enhancements

- **Forecast error analysis** - compare `demand_mw` against
  `demand_forecast_mw` to surface systematic prediction gaps by authority and
  hour of day.
- **Extended date range** - expand the pipeline to a full calendar year or
  multiple years for seasonality analysis.
- **Power BI implementation** - The planned Power BI implementation will use
  the existing dashboard data model.
- **Refined Stress Index** - replace the dataset-window peak with a rolling
  historical peak or seasonally adjusted baseline.
- **Anomaly detection** - apply a statistical threshold (z-score, rolling
  percentile) as an alternative or complement to the fixed classification tiers.
- **Geographic view** - map Stress Index by balancing authority service
  territory using a choropleth layer.

---

*This project uses public EIA-930 data and does not represent any utility's
internal systems, risk frameworks, or operational methodology.*